In [1]:
# Day 11 — AtmoSync Dashboard Analysis

## Objective

# Build an interactive dashboard from the processed AtmoSync telemetry, risk analytics, machine learning predictions, and spoilage arbitrage outputs.

In [2]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

PROCESSED_DIR = BASE_DIR / "data" / "processed"

monitoring_df = pd.read_csv(
    PROCESSED_DIR / "prediction_monitoring.csv"
)

telemetry_df = pd.read_csv(
    PROCESSED_DIR / "iot_telemetry_features.csv"
)

arbitrage_df = pd.read_csv(
    PROCESSED_DIR / "arbitrage_decisions.csv"
)

print("Monitoring:", monitoring_df.shape)
print("Telemetry:", telemetry_df.shape)
print("Arbitrage:", arbitrage_df.shape)

Monitoring: (1785, 12)
Telemetry: (1785, 35)
Arbitrage: (1785, 41)


In [4]:
## Dataset Overview

#The dashboard combines three major analytical outputs:

#- Prediction monitoring
#- Feature-engineered telemetry
#- Spoilage arbitrage decisions

In [5]:
print("Prediction Monitoring")
display(monitoring_df.head())

print("\nTelemetry Features")
display(telemetry_df.head())

print("\nArbitrage Decisions")
display(arbitrage_df.head())

Prediction Monitoring


,container_id,timestamp,commodity,temperature,humidity,vibration,condition,predicted_condition,prediction_confidence,confidence_category,prediction_status,monitoring_level
0,CONT_001,2026-09-10 17:39:32.831215,Avocado,3.94,71.79,0.29,WARNING,WARNING,0.995,HIGH CONFIDENCE,CORRECT,MONITOR
1,CONT_004,2026-09-10 17:39:34.852445,Mango,11.72,88.98,0.12,NORMAL,NORMAL,1.000,HIGH CONFIDENCE,CORRECT,NORMAL
2,CONT_005,2026-09-10 17:39:36.870612,Tomato,10.18,91.05,0.15,NORMAL,NORMAL,1.000,HIGH CONFIDENCE,CORRECT,NORMAL
3,CONT_005,2026-09-10 17:39:38.890192,Avocado,6.54,85.30,0.17,NORMAL,NORMAL,0.995,HIGH CONFIDENCE,CORRECT,NORMAL
4,CONT_005,2026-09-10 17:39:40.899424,Tomato,14.06,100.00,0.33,WARNING,WARNING,1.000,HIGH CONFIDENCE,CORRECT,MONITOR



Telemetry Features


,container_id,timestamp,commodity,temperature,humidity,vibration,condition,date,hour,day_of_week,...,max_vibration,warning_count,anomaly_count,avg_environmental_risk,commodity_avg_temperature,commodity_avg_humidity,commodity_avg_vibration,commodity_warning_count,commodity_anomaly_count,commodity_avg_risk
0,CONT_001,2026-09-10 17:39:32.831215,Avocado,3.94,71.79,0.29,WARNING,2026-09-10,17,3,...,0.59,62,36,2.894118,9.129163,88.956719,0.194729,71,53,2.952489
1,CONT_004,2026-09-10 17:39:34.852445,Mango,11.72,88.98,0.12,NORMAL,2026-09-10,17,3,...,0.59,81,36,2.942149,11.575468,86.437340,0.193234,97,51,2.689362
2,CONT_005,2026-09-10 17:39:36.870612,Tomato,10.18,91.05,0.15,NORMAL,2026-09-10,17,3,...,0.59,62,40,2.818182,10.869107,87.423878,0.184183,88,39,2.821351
3,CONT_005,2026-09-10 17:39:38.890192,Avocado,6.54,85.30,0.17,NORMAL,2026-09-10,17,3,...,0.59,62,40,2.818182,9.129163,88.956719,0.194729,71,53,2.952489
4,CONT_005,2026-09-10 17:39:40.899424,Tomato,14.06,100.00,0.33,WARNING,2026-09-10,17,3,...,0.59,62,40,2.818182,10.869107,87.423878,0.184183,88,39,2.821351



Arbitrage Decisions


,container_id,timestamp,commodity,temperature,humidity,vibration,condition,date,hour,day_of_week,...,commodity_avg_vibration,commodity_warning_count,commodity_anomaly_count,commodity_avg_risk,recommended_action,priority,intervention_required,arbitrage_opportunity,decision_reason,decision_score
0,CONT_001,2026-09-10 17:39:32.831215,Avocado,3.94,71.79,0.29,WARNING,2026-09-10,17,3,...,0.194729,71,53,2.952489,CONTINUE MONITORING,P4,False,False,elevated environmental risk,2
1,CONT_004,2026-09-10 17:39:34.852445,Mango,11.72,88.98,0.12,NORMAL,2026-09-10,17,3,...,0.193234,97,51,2.689362,CONTINUE MONITORING,P4,False,False,elevated environmental risk,2
2,CONT_005,2026-09-10 17:39:36.870612,Tomato,10.18,91.05,0.15,NORMAL,2026-09-10,17,3,...,0.184183,88,39,2.821351,INCREASE MONITORING,P3,False,False,elevated environmental risk,3
3,CONT_005,2026-09-10 17:39:38.890192,Avocado,6.54,85.30,0.17,NORMAL,2026-09-10,17,3,...,0.194729,71,53,2.952489,CONTINUE MONITORING,P4,False,False,elevated environmental risk,2
4,CONT_005,2026-09-10 17:39:40.899424,Tomato,14.06,100.00,0.33,WARNING,2026-09-10,17,3,...,0.184183,88,39,2.821351,CONSIDER REROUTING,P2,True,False,high humidity,6


In [6]:
print("Predicted Condition Distribution")

display(
    monitoring_df["predicted_condition"]
    .value_counts()
)

Predicted Condition Distribution


predicted_condition
NORMAL     1249
WARNING     352
ANOMALY     184
Name: count, dtype: int64

In [7]:
container_analysis = (
    monitoring_df
    .groupby("container_id")
    .agg(
        total_records=("container_id", "size"),
        anomalies=(
            "predicted_condition",
            lambda x: (x == "ANOMALY").sum()
        )
    )
    .reset_index()
)

container_analysis["anomaly_rate"] = (
    container_analysis["anomalies"]
    / container_analysis["total_records"]
    * 100
)

display(container_analysis)

,container_id,total_records,anomalies,anomaly_rate
0,CONT_001,340,36,10.588235
1,CONT_002,339,35,10.324484
2,CONT_003,369,37,10.027100
3,CONT_004,363,36,9.917355
4,CONT_005,374,40,10.695187


In [ ]:
## Dashboard Findings

#The dashboard summarizes:

#- Overall predicted condition distribution
#- Sensor trends
#- Spoilage risk
#- ML prediction confidence
#- Monitoring levels
#- Container-level anomaly rates
#- Commodity-level anomaly rates
#- Recommended operational actions

#All metrics are based on simulated telemetry and should not be interpreted as observed real-world spoilage outcomes.